# Разделение источников аудио — Визуализация

Этот ноутбук демонстрирует разделение источников аудио и визуализацию результатов.

## Настройка
Запустите следующую команду для установки зависимостей:

In [ ]:
# Install required packages
# !pip install -r requirements.txt

In [ ]:
import sys
sys.path.insert(0, '.')

## Загрузка и анализ аудио

In [ ]:
import torch
import torchaudio
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

SAMPLE_AUDIO = "data/test/sample.wav"  # Замените на ваш аудиофайл

In [ ]:
def load_audio(path):
    """Загрузка аудиофайла"""
    waveform, sr = torchaudio.load(path)
    return waveform, sr

def plot_waveform(waveform, sr, title="Waveform"):
    """Построение графика волновой формы"""
    plt.figure(figsize=(12, 4))
    
    if waveform.shape[0] > 1:
        for i in range(waveform.shape[0]):
            plt.plot(waveform[i].numpy(), label=f"Channel {i+1}")
    else:
        plt.plot(waveform[0].numpy())
    
    plt.title(title)
    plt.xlabel("Samples")
    plt.ylabel("Amplitude")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

def plot_spectrogram(waveform, sr, title="Spectrogram", n_fft=1024):
    """Plot spectrogram using STFT."""
    spec = torchaudio.functional.spectrogram(
        waveform, n_fft=n_fft
    )
    
    plt.figure(figsize=(12, 6))
    plt.imshow(
        torch.log10(spec[0] + 1e-10).numpy(),
        origin="lower",
        aspect="auto",
        cmap="viridis"
    )
    plt.title(title)
    plt.xlabel("Time")
    plt.ylabel("Frequency (Hz)")
    plt.colorbar(label="log magnitude")
    plt.show()

## Анализ одного трека

In [ ]:
# Загрузка и анализ образца аудио
if Path(SAMPLE_AUDIO).exists():
    waveform, sr = load_audio(SAMPLE_AUDIO)
    duration = waveform.shape[1] / sr
    
    print(f"Sample Rate: {sr} Hz")
    print(f"Duration: {duration:.2f} seconds")
    print(f"Channels: {waveform.shape[0]}")
    print(f"Samples: {waveform.shape[1]}")
    
    plot_waveform(waveform, sr, "Original Audio Waveform")
    plot_spectrogram(waveform, sr, "Original Audio Spectrogram")
else:
    print(f"Sample file not found: {SAMPLE_AUDIO}")
    print("Please add a test audio file to data/test/")

## Запуск разделения источников

In [ ]:
from app.separator import SourceSeparator

# Инициализация разделителя
separator = SourceSeparator(
    model_name="htdemucs_ft",
    device="cpu",
    output_dir="output"
)

print("Separator initialized")

In [ ]:
# Запуск разделения
if Path(SAMPLE_AUDIO).exists():
    stems = separator.separate(SAMPLE_AUDIO)
    
    print("\nSeparated stems:")
    for name, path in stems.items():
        print(f"  {name}: {path}")
else:
    print("No sample audio to process")

## Визуализация разделённых стемов

In [ ]:
# Визуализация каждого стема
if Path(SAMPLE_AUDIO).exists():
    stems_dir = Path("output") / Path(SAMPLE_AUDIO).stem
    
    stem_names = ["drums", "bass", "other", "vocals"]
    
    for stem_name in stem_names:
        stem_path = stems_dir / f"{stem_name}.wav"
        
        if stem_path.exists():
            stem_wave, stem_sr = load_audio(str(stem_path))
            
            plot_waveform(
                stem_wave, 
                stem_sr, 
                f"{stem_name.upper()} - Waveform"
            )
            plot_spectrogram(
                stem_wave, 
                stem_sr, 
                f"{stem_name.upper()} - Spectrogram"
            )

## Расчёт метрик

In [ ]:
from app.metrics import SeparationMetrics

calculator = SeparationMetrics()
print("Metrics calculator initialized")

In [ ]:
# Расчёт метрик, если доступен эталонный трек
# results = calculator.evaluate_full_track(
#     reference_dir=Path("data/test/track_name"),
#     estimated_dir=Path("output/track_name")
# )
# 
# print("\n=== Метрики ===")
# for source, metrics in results.items():
#     print(f"\n{source.upper()}:")
#     for metric, value in metrics.items():
#         print(f"  {metric}: {value:.2f} дБ")

## Итоги

Этот ноутбук демонстрирует:
- Загрузку и визуализацию волновых форм и спектрограмм аудио
- Запуск разделения источников с помощью Demucs
- Визуализацию разделённых стемов
- Расчёт метрик качества

## Примечания
- Время обработки зависит от оборудования (CPU/GPU)
- Для наилучшего качества используйте модель htdemucs_ft
- Для более быстрой обработки используйте короткие аудиоклипы для тестирования